# ADAN Trading Bot - Colab Training v2
## Robust Setup with Real-time Monitoring

This notebook sets up and trains the ADAN Trading Bot on Google Colab with:
- ✅ Automatic dependency installation
- ✅ Project package installation
- ✅ Real-time training monitoring
- ✅ Model extraction and download
- ✅ Error handling and recovery

## Step 1: Setup Environment

In [ ]:
import os
import sys
from pathlib import Path

# Check if running on Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("⚠ Not running on Google Colab")

# Display GPU info
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU available"

## Step 2: Clone Repository and Install Dependencies

In [ ]:
%%bash
set -e

echo "📦 Installing system dependencies..."
apt-get update -qq
apt-get install -y -qq build-essential python3-dev wget > /dev/null 2>&1

echo "📦 Installing TA-Lib C library..."
cd /tmp
wget -q https://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.28-src.tar.gz
tar -xzf ta-lib-0.4.28-src.tar.gz
cd ta-lib
./configure > /dev/null 2>&1
make > /dev/null 2>&1
make install > /dev/null 2>&1
ldconfig
cd /tmp && rm -rf ta-lib ta-lib-0.4.28-src.tar.gz

echo "✓ System dependencies installed"

In [ ]:
%%bash
set -e

echo "📦 Cloning ADAN repository..."
if [ ! -d "/content/ADAN0" ]; then
    cd /content
    git clone https://github.com/Cabrel10/ADAN0.git
else
    cd /content/ADAN0
    git pull origin main
fi

echo "✓ Repository ready"
ls -lh /content/ADAN0/data/*.parquet 2>/dev/null | head -3 || echo "⚠ Data files not found yet"

In [ ]:
%%bash
set -e

echo "📦 Installing Python dependencies..."
pip install --upgrade pip setuptools wheel -q

# Install with compatible versions
pip install --no-cache-dir -q \
    "numpy>=1.24.0,<2.0.0" \
    "pandas>=2.0.0" \
    "scipy>=1.10.0" \
    "scikit-learn>=1.3.0" \
    "matplotlib>=3.7.0" \
    "yfinance>=0.2.0" \
    "pandas-ta>=0.3.14b0" \
    "torch>=2.0.0" \
    "gymnasium>=0.29.0" \
    "stable-baselines3>=2.0.0" \
    "optuna>=3.0.0" \
    "pyyaml>=6.0" \
    "tqdm>=4.65.0"

echo "📦 Installing TA-Lib Python wrapper..."
pip install --no-cache-dir -q TA-Lib

echo "✓ Python dependencies installed"

In [ ]:
%%bash
set -e

cd /content/ADAN0

echo "📦 Installing ADAN project (editable mode)..."
pip install --no-cache-dir -e . -q

echo "✓ ADAN project installed"

## Step 3: Verify Installation

In [ ]:
import sys

print("🔍 Verifying imports...\n")

errors = []

# Test critical imports
imports_to_test = [
    ("adan_trading_bot", "ADAN Trading Bot"),
    ("adan_trading_bot.environment.multi_asset_chunked_env", "MultiAssetChunkedEnv"),
    ("stable_baselines3", "Stable Baselines 3"),
    ("optuna", "Optuna"),
    ("torch", "PyTorch"),
    ("gymnasium", "Gymnasium"),
]

for module_name, display_name in imports_to_test:
    try:
        __import__(module_name)
        print(f"✓ {display_name}")
    except ImportError as e:
        print(f"✗ {display_name}: {e}")
        errors.append((display_name, str(e)))

if errors:
    print(f"\n❌ {len(errors)} import(s) failed!")
    for name, error in errors:
        print(f"  - {name}: {error}")
    sys.exit(1)
else:
    print("\n✅ All imports verified successfully!")

## Step 4: Configure Training Parameters

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    'timesteps': 500000,  # Total training steps
    'save_freq': 50000,   # Save checkpoint every N steps
    'eval_freq': 10000,   # Evaluate every N steps
    'n_workers': 4,       # Parallel workers
    'config_path': '/content/ADAN0/config/config.yaml',
    'checkpoint_dir': '/content/ADAN0/checkpoints',
    'log_dir': '/content/ADAN0/logs',
}

print("📋 Training Configuration:")
print(f"  Timesteps: {TRAINING_CONFIG['timesteps']:,}")
print(f"  Workers: {TRAINING_CONFIG['n_workers']}")
print(f"  Save frequency: {TRAINING_CONFIG['save_freq']:,} steps")
print(f"  Eval frequency: {TRAINING_CONFIG['eval_freq']:,} steps")
print(f"  Config: {TRAINING_CONFIG['config_path']}")
print(f"  Checkpoints: {TRAINING_CONFIG['checkpoint_dir']}")

## Step 5: Start Training with Monitoring

In [ ]:
import subprocess
import time
from datetime import datetime

os.chdir('/content/ADAN0')

print("🚀 Starting training...")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Build command
cmd = [
    'python', 'scripts/train_parallel_agents.py',
    '--config-path', TRAINING_CONFIG['config_path'],
    '--checkpoint-dir', TRAINING_CONFIG['checkpoint_dir'],
    '--timesteps', str(TRAINING_CONFIG['timesteps']),
    '--resume',
]

# Run training
try:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1
    )
    
    # Stream output
    for line in process.stdout:
        print(line.rstrip())
    
    returncode = process.wait()
    
    print("="*60)
    if returncode == 0:
        print(f"✅ Training completed successfully!")
        print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    else:
        print(f"❌ Training failed with return code: {returncode}")
        
except KeyboardInterrupt:
    print("\n⚠ Training interrupted by user")
    process.terminate()
except Exception as e:
    print(f"❌ Error during training: {e}")

## Step 6: Extract and Download Model

In [ ]:
from pathlib import Path

print("📦 Extracting trained model...")

# Run extraction script
cmd = [
    'python', 'scripts/extract_model.py',
    '--latest',
    '--output', 'models/extracted'
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/ADAN0')
    print(result.stdout)
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
except Exception as e:
    print(f"Error extracting model: {e}")

In [ ]:
# List extracted files
extracted_dir = Path('/content/ADAN0/models/extracted')

if extracted_dir.exists():
    print("✓ Extracted model files:")
    for file in sorted(extracted_dir.glob('*')):
        size = file.stat().st_size / 1024 / 1024  # MB
        print(f"  - {file.name} ({size:.1f} MB)")
else:
    print("⚠ No extracted model directory found")

## Step 7: Download Results to Local Machine

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    print("📥 Preparing files for download...")
    
    # Create archive with results
    import shutil
    
    archive_path = '/tmp/adan_results'
    os.makedirs(archive_path, exist_ok=True)
    
    # Copy extracted model
    if extracted_dir.exists():
        shutil.copytree(
            extracted_dir,
            Path(archive_path) / 'extracted_model',
            dirs_exist_ok=True
        )
        print("✓ Copied extracted model")
    
    # Copy logs
    logs_dir = Path('/content/ADAN0/logs')
    if logs_dir.exists():
        shutil.copytree(
            logs_dir,
            Path(archive_path) / 'logs',
            dirs_exist_ok=True
        )
        print("✓ Copied logs")
    
    # Create archive
    archive_name = 'adan_training_results'
    shutil.make_archive(archive_name, 'zip', archive_path)
    print(f"✓ Created archive: {archive_name}.zip")
    
    # Download
    print("\n📥 Downloading results...")
    files.download(f'{archive_name}.zip')
    print("✅ Download complete!")
else:
    print("⚠ Not running on Colab, skipping download")

## Step 8: Summary

In [ ]:
print("\n" + "="*60)
print("✅ ADAN Training Complete!")
print("="*60)
print("\nResults:")
print(f"  - Checkpoints: {TRAINING_CONFIG['checkpoint_dir']}")
print(f"  - Extracted model: {extracted_dir}")
print(f"  - Logs: {TRAINING_CONFIG['log_dir']}")
print("\nNext steps:")
print("  1. Download the results archive")
print("  2. Extract model files locally")
print("  3. Use model.zip for inference")
print("  4. Review logs for training metrics")
print("="*60)